In [7]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental ddgs langchain-experimental

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


***BUILT IN TOOLS***

**DuckDuckGo Search**

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun
search=DuckDuckGoSearchRun()
result=search.invoke("CURRENT CRICKET NEWS")
print(result)

1 week ago - Cricket News · Anushka, Bhatia to captain India A teams in home series against Australia A · Sixteen-year-old Deeya Yadav not picked in any squads after playing T20s against England A · 27-Aug-2026 • 2 hrs ago•Cricinfo staff · SA domestic ... 3 days ago - Accidental IND opener averaged 53 in 6 Tests but left after Ganguly chose YuvrajHis action became a World Cup symbol; his Test career ended only after two gamesPCB raised concerns, BCCI followed up as alarm grows over Asiad cricket venueTough days ahead: How the Colombo draw has dented India’s chances for WTC finalDravid and Sehwag back in Indian team: Aryavir, Anvay named in U-19 squadPCB complains about Michael Atherton's interview with Imran Khan's sonsSiraj fails to conquer SL conditions: In Bumrah's absence, he disappoints‘Bit of confusion’: ABD says Sachin's record better than Kohli but stats not all‘Looking like Tendulkar’: Heavy praise for Sonal Dinusha after he makes merry‘Imran Khan in the dark most of the time’

**Shell Tool** (hands that can interact with the computer)

In [9]:
from langchain_community.tools import ShellTool
shell_tool=ShellTool()
result=shell_tool.invoke("ls")
print(result)

Executing command:
 ls
01_TOOLS.ipynb



/home/udit-tripathi/.local/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


***CUSTOM TOOLS***

**1. USING @tool DECORATOR**

In [13]:
from langchain_core.tools import tool

In [14]:
# STEP 1- CREATE A FUNCTION
def multiply(a,b):
    """Multiply 2 numbers"""
    return a*b

In [15]:
# STEP 2 - ADD TYPE HINTS
def multiply(a:int,b:int)-> int:
    """Multiply 2 numbers"""
    return a*b

In [16]:
# STEP 3 - ADD TOOL DECORATOR
@tool 
def multiply(a:int,b:int)->int:
    """Multiply 2 numbers"""
    return a*b

In [17]:
result=multiply.invoke({"a":4,"b":3})
print(result)

12


In [19]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [20]:
print(multiply.args_schema.model_json_schema())  # this is what llm receives in order to give output

{'description': 'Multiply 2 numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


**2. USING STRUCTURED TOOL & PYDANTIC**

In [23]:
from langchain_core.tools import StructuredTool
from pydantic import Field,BaseModel

In [25]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True,description="The first num to multiply")
    b: int = Field(required=True,description="The second num to multiply")

/tmp/ipykernel_27373/1457652784.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True,description="The first num to multiply")
/tmp/ipykernel_27373/1457652784.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True,description="The second num to multiply")


In [26]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [27]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [29]:
result=multiply.invoke({"a":4,"b":3})
print(result)

print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first num to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second num to multiply', 'title': 'B', 'type': 'integer'}}


**3. USING BASE TOOL CLASS**

*BaseTool is an abstract/base class in LangChain that defines the common interface for a tool.*

*BaseTool is a blueprint for tools that an LLM/agent can call.*

In [30]:
from langchain.tools import BaseTool
from typing import Type

In [31]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_27373/3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_27373/3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [32]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [33]:
multiply_tool = MultiplyTool()

In [34]:
result = multiply_tool.invoke({'a':4, 'b':3})

print(result)

print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


***TOOLKITS***

*Toolkit = a collection of related tools.*

In [35]:
from langchain_core.tools import tool

In [36]:
@tool
def mult(a:int,b:int)->int:
    """Multiply 2 numbers"""
    return a*b

@tool
def add(a:int,b:int)->int:
    """Add 2 numbers"""
    return a+b

In [39]:
class MathToolKit:
    def get_tools(self):
        return [mult,add]
    

In [40]:
toolkit=MathToolKit()
tools=toolkit.get_tools()

for tool in tools:
    print(tool.name,">>",tool.description)

mult >> Multiply 2 numbers
add >> Add 2 numbers
